# Instances, solutions, and submission archives

Solutions are submitted as a single ZIP archive. This notebook walks that path
end to end: reading the instances, checking your solutions against them before
you upload anything, and packing them up.

Two archives ship with this repository so that everything below runs on real
data: `example_instances.zip` and `example_solutions.zip`, which holds one
feasible solution for each of those instances.

Those solutions are correct but far from good.

## The two archives

`example_instances.zip` is a copy of the example instances, also available at

<https://cgshop.ibr.cs.tu-bs.de/content_management/media/cgshop_2027_example_instances>

Note that the archive may have been updated; the solutions in `example_solutions.zip` are for the instances in `example_instances.zip` in this repository.

In [1]:
from pathlib import Path

instance_zip = Path("example_instances.zip")
solution_zip = Path("example_solutions.zip")

for archive in (instance_zip, solution_zip):
    print(f"{archive} ({archive.stat().st_size / 1000:.0f} kB)")

example_instances.zip (513 kB)
example_solutions.zip (111 kB)


## Reading the instances

`InstanceDatabase` reads instances directly out of an archive without unpacking.
It can also work on a directory.
A lookup key may be the bare name, the file name, or a path to it.

Pass `enable_cache=True` when you look instances up repeatedly: by default every
`db[uid]` re-reads and re-parses the member, which can cost more than the
verification itself.

In [3]:
from cgshop2027_pyutils.instance_database import InstanceDatabase

db = InstanceDatabase(instance_zip, enable_cache=True)

uids = [instance.instance_uid for instance in db]
print(len(uids), "instances, e.g.", uids[:3])

instance = db["srpg_246_821_1s3"]
holes = len(instance.region_to_cover.inner_boundaries)
print(f"{instance.instance_uid}: {instance.number_of_cutters} cutters, {holes} holes")

75 instances, e.g. ['fpg73_100k_1s2', 'fpg73_100k_2s2', 'fpg73_100k_3s2']
srpg_246_821_1s3: 1 cutters, 0 holes


## Reading solutions out of an archive

`ZipSolutionIterator` is the reader and already checks parts of the zip archive,
performing path traversal, member and total size limits and CRC checks.
It also validates every member against the solution schema before handing it to you,
and records where each solution came from in `meta["zip_info"]`.

If you want to avoid any issues on upload, point it at your own solution archive to find out early whether it can be read at all before uploading.

In [4]:
from cgshop2027_pyutils.zip import ZipSolutionIterator

first = next(iter(ZipSolutionIterator(solution_zip)))
objective = first.max_tour_length
print(f"{first.instance_uid}: {len(first.tours)} tours, objective {objective}")
print(first.meta["zip_info"])

fpg73_100k_1s2: 1 tours, objective 98540
{'zip_file': 'example_solutions.zip', 'file_in_zip': 'fpg73_100k_1s2.solution.json'}


## Checking your solutions before you submit

`SolutionValidator.check_for_errors` returns a list of error messages rather than raising an exception;
an empty list means the solution is feasible for that instance.

Build one `SolutionValidator` per instance and keep it;
its constructor rasterizes the region and the cutter,
which is the expensive half of the work and depends only on the instance.
Checking all 75 example solutions this way takes a fraction of a second,
so there is no reason to submit an archive you have not checked yourself.

In [5]:
import time

from cgshop2027_pyutils.verify import SolutionValidator

validators: dict[str, SolutionValidator] = {}
objectives: dict[str, int] = {}
rejected: list[tuple[str, list[str]]] = []

started = time.perf_counter()
for solution in ZipSolutionIterator(solution_zip):
    instance = db[solution.instance_uid]
    validator = validators.get(instance.instance_uid)
    if validator is None:
        validator = validators[instance.instance_uid] = SolutionValidator(instance)
    errors = validator.check_for_errors(solution)
    if errors:
        rejected.append((solution.instance_uid, errors))
    else:
        objectives[solution.instance_uid] = solution.max_tour_length
elapsed = time.perf_counter() - started

print(f"{len(objectives)} feasible, {len(rejected)} with errors, in {elapsed:.2f}s")

75 feasible, 0 with errors, in 0.34s


## The objective

The objective is `max_tour_length`, the length of the longest tour, and it is to
be minimized. These numbers should be relatively easy to beat, since the example solutions are constructed using very simple heuristics.

In [6]:
for uid in uids:
    print(f"  {uid:<20} {objectives[uid]:>8}")

print(f"\nsmallest: {min(objectives.values())}, largest: {max(objectives.values())}")

  fpg73_100k_1s2          98540
  fpg73_100k_2s2          49462
  fpg73_100k_3s2          33230
  fpg73_1k_1s2              902
  fpg73_1k_2s2              464
  fpg73_1k_3s2              320
  fpg73_20k_1s2            9472
  fpg73_20k_2s2            4872
  fpg73_20k_3s2            3188
  iso_11k_6k_3p             914
  iso_145k_26k_3s3         5762
  iso_191_11k_2l1          2112
  iso_260k_15k_1d6         4184
  iso_567_6466_3l1         1248
  iso_725k_1813_3d6         250
  iso_7k_11k_1s4           2232
  isoa_1371_100k_3s2      38048
  isoa_13m_98k_3l1        15906
  isoa_143k_3k_2l1          958
  isoa_28k_1203_1l1         542
  octa_1545_100k_1d6      32336
  octa_189k_10k_3p         1982
  octa_189k_4k_2s2         1936
  octa_25k_26k_1s3        16916
  octa_25k_30k_3l1         4618
  octa_32k_3k_1s4          1428
  octa_34k_1504_2p          538
  octa_34k_31k_3l1         4866
  octa_381k_40k_2s2       15868
  octa_3k_3k_1s2           2962
  pla34k_75k_1p           24738
  pla34k

## What an infeasible solution looks like

The tours themselves are checked by the schema as a solution is parsed: they must
be closed, axis-parallel, and must not repeat a point immediately. 

The remaining feasibility checks need the instance:
there must be exactly one trajectory per available cutter,
and together they must sweep the whole region.
Miss a cell and see the verifier report it as missing:

In [7]:
from cgshop2027_pyutils.schemas import CGSHOP2027Solution

parked = CGSHOP2027Solution(
    instance_uid="srpg_246_821_1s3", tours=[{"x": [0], "y": [0]}]
)
for error in SolutionValidator(db["srpg_246_821_1s3"]).check_for_errors(parked):
    print(error)

The region is not covered: 821 cells are never swept, for example the cell at (35, 1).


A broken *archive* is a different matter: the iterator raises and nothing in it
can be trusted. `ZipReaderError` covers the archive-level problems (bad file
names, oversized members, corrupt data, no solutions found at all) and
`BadSolutionFile` a member that is not a valid solution, naming the file it
choked on.

In [8]:
import tempfile
import zipfile

from cgshop2027_pyutils.zip import BadSolutionFile, ZipReaderError

broken = Path(tempfile.mkdtemp()) / "broken.zip"
with zipfile.ZipFile(broken, "w") as archive:
    archive.writestr("demo.solution.json", '{"not": "a solution"}')

try:
    list(ZipSolutionIterator(broken))
except (ZipReaderError, BadSolutionFile) as error:
    print(f"{type(error).__name__}: {str(error).splitlines()[0]}")

BadSolutionFile: Error in file 'demo.solution.json': 2 validation errors for CGSHOP2027Solution


## Packing your own archive

`ZipWriter` is the other direction. It names each member after the solution's
`instance_uid`, following the `NAME.solution.json` convention, and refuses to
overwrite an existing file. Feed it the solutions your solver produced and
submit what comes out.

In [9]:
from cgshop2027_pyutils.zip import ZipWriter

my_submission = Path(tempfile.mkdtemp()) / "solutions.zip"
with ZipWriter(my_submission) as writer:
    for solution in ZipSolutionIterator(solution_zip):
        writer.add_solution(solution)

with zipfile.ZipFile(my_submission) as archive:
    print(len(archive.namelist()), "members, e.g.", archive.namelist()[:2])

75 members, e.g. ['fpg73_100k_1s2.solution.json', 'fpg73_100k_2s2.solution.json']
